In [1]:
from mainnet_launch.pages.asset_discounts.stable_coin_pricing import (
    StableCoinConsants,
    stablecoin_constants,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    get_state_by_one_block,
    build_blocks_to_use,
    _build_blocks_to_use_dont_clip,
)
from mainnet_launch.constants import ETH_CHAIN
import plotly.express as px
import plotly.io as pio
import pandas as pd

pio.templates.default = None
token_symbols = [c.symbol for c in stablecoin_constants if c.symbol not in ["sUSDs", "USDs"]]


def fetch_stable_coin_spot_safe_and_backing():
    backing_calls = [c.backing_call for c in stablecoin_constants]
    safe_price_calls = [c.safe_price_call for c in stablecoin_constants if c.safe_price_call is not None]
    # blocks = _build_blocks_to_use_dont_clip(ETH_CHAIN, start_block=21389810, end_block=ETH_CHAIN.client.eth.block_number, approx_num_blocks_per_day=12)
    blocks = build_blocks_to_use(ETH_CHAIN, start_block=21389810)
    df = get_raw_state_by_blocks([*backing_calls, *safe_price_calls], blocks, ETH_CHAIN, include_block_number=True)
    df["sUSDs_safe_price"] = df["sUSDs_backing"] * df["USDs_safe_price"]
    df["scrvUSD_safe_price"] = df["scrvUSD_backing"] * df["crvUSD_safe_price"]
    df["sUSDe_safe_price"] = df["sUSDe_backing"] * df["USDe_safe_price"]

    # # 100 * (peg - safe) / peg
    # for symbol in token_symbols:
    #     df[f"{symbol}_discount"] = 100 * ((df[f"{symbol}_backing"] - df[f"{symbol}_safe_price"]) / df[f"{symbol}_backing"])

    discount_cols = [c for c in df.columns if "_discount" in c]
    px.line(df[discount_cols], title="stablecoin percent discoutns")

    spot_price_calls = [
        call
        for calls in [c.spot_price_calls for c in stablecoin_constants if c.spot_price_calls is not None]
        for call in calls
    ]
    spot_price_df = get_raw_state_by_blocks(spot_price_calls, blocks, ETH_CHAIN)
    full_df = pd.concat([df, spot_price_df], axis=1)

    full_df["DAI_spot_price"] = full_df[["DAI_to_USDC_spot_price", "DAI_to_USDC_spot_price2"]].max(axis=1)
    full_df["USDs_spot_price"] = full_df["DAI_spot_price"]  # dai is 1:1 with USDs

    full_df["USDe_to_USDC_spot_price2"] = full_df["sUSEe_to_USDC_spot_price"].divide(full_df["sUSDe_backing"])
    full_df["USDe_spot_price"] = full_df[["USDe_to_USDC_spot_price", "USDe_to_USDC_spot_price2"]].max(axis=1)

    full_df["USDC_spot_price"] = 1.0  # by definition

    full_df["USDT_spot_price"] = full_df[["USDT_to_USDC_spot_price", "USDT_to_USDC_spot_price2"]].max(axis=1)
    full_df["GHO_spot_price"] = full_df[["GHO_to_USDC_spot_price"]].max(axis=1)
    full_df["crvUSD_to_USDC_spot_price2"] = full_df["crvUSD_to_USDT_spot_price"] * full_df["USDT_spot_price"]

    full_df["crvUSD_spot_price"] = full_df[["crvUSD_to_USDC_spot_price", "crvUSD_to_USDC_spot_price2"]].max(axis=1)

    full_df["scrvUSD_spot_price"] = full_df["crvUSD_spot_price"] * full_df["scrvUSD_backing"]
    full_df["sUSDe_spot_price"] = full_df["USDe_spot_price"] * full_df["sUSDe_backing"]

    return full_df


df = fetch_stable_coin_spot_safe_and_backing()

df

Inserted new rows into 'BLOCKS_TO_USE_TABLE' while avoiding duplicates.
Table 'BLOCKS_TO_USE_TABLE'already exists and data inserted.
Successfully updated table_name='BLOCKS_TO_USE_TABLE' current_time=datetime.datetime(2025, 3, 13, 19, 12, 28, 52392, tzinfo=datetime.timezone.utc).
Inserted new rows into 'BLOCKS_TO_USE_TABLE' while avoiding duplicates.
Table 'BLOCKS_TO_USE_TABLE'already exists and data inserted.
Successfully updated table_name='BLOCKS_TO_USE_TABLE' current_time=datetime.datetime(2025, 3, 13, 19, 12, 29, 20082, tzinfo=datetime.timezone.utc).


,DAI_backing,USDe_backing,USDC_backing,USDT_backing,USDs_backing,GHO_backing,crvUSD_backing,scrvUSD_backing,sUSDe_backing,sUSDs_backing,...,USDs_spot_price,USDe_to_USDC_spot_price2,USDe_spot_price,USDC_spot_price,USDT_spot_price,GHO_spot_price,crvUSD_to_USDC_spot_price2,crvUSD_spot_price,scrvUSD_spot_price,sUSDe_spot_price
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-12-12 23:48:23+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.020908,1.134721,1.016792,...,0.999911,0.998915,1.001425,1.0,1.000201,0.996512,0.998708,0.998708,1.019589,1.136338
2024-12-13 23:55:59+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.021230,1.135382,1.017122,...,0.999979,0.999192,1.001334,1.0,1.000151,0.996492,0.998549,0.998797,1.020001,1.136896
2024-12-14 23:32:47+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.021539,1.136025,1.017445,...,0.999925,0.998883,1.000927,1.0,0.999837,0.998313,0.998892,0.999027,1.020545,1.137078
2024-12-15 23:40:47+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.021881,1.136681,1.017775,...,0.999929,0.998501,1.000932,1.0,0.999710,0.999005,0.999155,0.999155,1.021017,1.137740
2024-12-16 23:46:11+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.022238,1.137332,1.018105,...,0.999909,0.997086,1.001211,1.0,1.000096,0.998940,0.999306,0.999306,1.021529,1.138709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-08 23:53:59+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.041660,1.161639,1.042103,...,0.999861,0.994722,0.999284,1.0,0.999692,0.999158,0.999578,0.999663,1.041309,1.160807
2025-03-09 23:31:47+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.041660,1.161793,1.042280,...,0.999821,0.994101,0.999561,1.0,0.999693,0.999383,0.999541,0.999799,1.041451,1.161283
2025-03-10 23:40:35+00:00,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.041671,1.161952,1.042461,...,0.999792,0.993421,0.998997,1.0,0.999425,0.999005,0.999325,0.999437,1.041084,1.160787


In [2]:
safe_price_cols = [f"{t}_safe_price" for t in token_symbols]
backing_cols = [f"{t}_backing" for t in token_symbols]
spot_price_cols = [f"{t}_spot_price" for t in token_symbols]
df[[*spot_price_cols, *spot_price_cols, *backing_cols]].to_csv("stable_coin_prices.csv")

percent_difference_from_peg_in_safe = pd.DataFrame(
    data=100 * (df[backing_cols].values - df[safe_price_cols].values) / (df[backing_cols].values),
    columns=token_symbols,
    index=df.index,
)
percent_difference_from_peg_in_safe.to_csv("percent_difference_from_peg_in_safe.csv")

percent_difference_between_safe_and_spot = pd.DataFrame(
    data=100 * (df[safe_price_cols].values - df[spot_price_cols].values) / (df[safe_price_cols].values),
    columns=token_symbols,
    index=df.index,
)
percent_difference_between_safe_and_spot.to_csv("percent_difference_between_safe_and_spot.csv")

percent_difference_from_peg_in_spot = pd.DataFrame(
    data=100 * (df[backing_cols].values - df[spot_price_cols].values) / (df[backing_cols].values),
    columns=token_symbols,
    index=df.index,
)
percent_difference_from_peg_in_spot.to_csv("percent_difference_from_peg_in_spot.csv")

px.line(percent_difference_from_peg_in_spot, title="100 * (backing - spot_price) / backing").show()
px.line(percent_difference_from_peg_in_safe, title="100 * (backing - safe price) / backing").show()
px.line(percent_difference_between_safe_and_spot, title="100 * (safe price - spot_price) / spot price").show()

In [3]:
px.ecdf(percent_difference_from_peg_in_spot, title="100 * (backing - spot_price) / backing").show()
px.ecdf(percent_difference_from_peg_in_safe, title="100 * (backing - safe price) / backing").show()
px.ecdf(percent_difference_between_safe_and_spot, title="100 * (safe price - spot_price) / spot price").show()

In [16]:
percent_difference_between_safe_and_spot.describe(percentiles=[0.2, 0.8]).T.round(4) * 100

,count,mean,std,min,20%,50%,80%,max
DAI,9100.0,1.34,1.63,-1.92,-0.03,1.08,2.84,5.98
USDe,9100.0,1.38,5.21,-17.86,-1.35,1.85,4.56,14.21
USDC,9100.0,-0.47,0.50,-1.91,-0.94,-0.40,0.00,0.40
USDT,9100.0,0.58,3.77,-12.24,-2.14,0.78,3.66,9.40
GHO,9100.0,6.19,3.57,-6.63,3.85,6.00,8.52,15.45
crvUSD,9100.0,1.23,3.66,-8.94,-1.18,0.79,3.64,13.80
scrvUSD,9100.0,1.23,3.66,-8.94,-1.18,0.79,3.64,13.80
sUSDe,9100.0,1.38,5.21,-17.86,-1.35,1.85,4.56,14.21


In [15]:
px.ecdf(percent_difference_between_safe_and_spot)

In [5]:
from mainnet_launch.constants import ETH_CHAIN, ROOT_PRICE_ORACLE

from multicall import Call

from mainnet_launch.data_fetching.get_state_by_block import (
    safe_normalize_6_with_bool_success,
    safe_normalize_with_bool_success,
)

USDC = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"


def build_root_price_oracle_to_USDC_call(token_address: str, name: str) -> Call:
    return Call(
        ROOT_PRICE_ORACLE(ETH_CHAIN),
        ["getPriceInQuote(address,address)(uint256)", ETH_CHAIN.client.toChecksumAddress(token_address), USDC],
        [(name, safe_normalize_6_with_bool_success)],
    )


calls = [
    build_root_price_oracle_to_USDC_call(c.token_address, f"{c.token_address} {c.symbol}") for c in stablecoin_constants
]
get_state_by_one_block(calls, 22033308 + 1, ETH_CHAIN)
get_state_by_one_block(calls, ETH_CHAIN.client.eth.block_number, ETH_CHAIN)

{'0x6B175474E89094C44Da98b954EedeAC495271d0F DAI': 0.993894,
 '0x4c9EDD5852cd905f086C759E8383e09bff1E68B3 USDe': None,
 '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48 USDC': 1.0,
 '0xdAC17F958D2ee523a2206206994597C13D831ec7 USDT': 0.997027,
 '0xdC035D45d973E3EC169d2276DDab16f1e407384F USDs': 0.998549,
 '0x40D16FC0246aD3160Ccc09B8D0D3A2cD28aE6C2f GHO': 0.998386,
 '0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E crvUSD': 0.99859,
 '0x0655977FEb2f289A4aB78af67BAB0d17aAb84367 scrvUSD': 1.040253,
 '0x9D39A5DE30e57443BfF2A8307A4256c8797A3497 sUSDe': 1.157795,
 '0xa3931d71877C0E7a3148CB7Eb4463524FEc27fbD sUSDs': 1.041454}

In [6]:
stablecoin_constants

[StableCoinConsants(token_address='0x6B175474E89094C44Da98b954EedeAC495271d0F', symbol='DAI', decimals=18, backing_call=<Call dummy()(uint256) on 0x000000>, safe_price_call=<Call latestRoundData()((uint80,int128,uint256,uint256,uint80)) on 0xAed0c3>, spot_price_calls=[<Call get_dy(int128,int128,uint256)(uint256) on 0xbEbc44>, <Call get_dy(int128,int128,uint256)(uint256) on 0xA5407e>]),
 StableCoinConsants(token_address='0x4c9EDD5852cd905f086C759E8383e09bff1E68B3', symbol='USDe', decimals=18, backing_call=<Call dummy()(uint256) on 0x000000>, safe_price_call=<Call latestRoundData()((uint80,int128,uint256,uint256,uint80)) on 0xa569d9>, spot_price_calls=[<Call get_dy(int128,int128,uint256)(uint256) on 0x029504>, <Call querySwap((bytes32,uint8,address,address,uint256,bytes),(address,bool,address,bool))(uint256) on 0xE39B5e>]),
 StableCoinConsants(token_address='0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', symbol='USDC', decimals=6, backing_call=<Call dummy()(uint256) on 0x000000>, safe_pric

In [7]:
ROOT_PRICE_ORACLE(ETH_CHAIN)

'0x61F8BE7FD721e80C0249829eaE6f0DAf21bc2CaC'

In [8]:
pass

In [9]:
DAI_to_USDC_spot_price = Call(
    "0xbEbc44782C7dB0a1A60Cb6fe97d0b483032FF1C7",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("DAI_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)

DAI_to_USDC_spot_price2 = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("DAI_to_USDC_spot_price2", safe_normalize_6_with_bool_success)],
)


USDT_to_USDC_spot_price = Call(
    "0xbEbc44782C7dB0a1A60Cb6fe97d0b483032FF1C7",
    ["get_dy(int128,int128,uint256)(uint256)", 2, 1, int(1e6)],
    [("USDT_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)

USDT_to_USDC_spot_price_2 = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 2, 1, int(168)],
    [("USDT_to_USDC_spot_price_2", safe_normalize_6_with_bool_success)],
)


crvUSD_to_USDC_spot_price = Call(
    "0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E",
    ["get_dy(int128,int128,uint256)(uint256)", 1, 0, int(1e18)],
    [("crvUSD_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


crvUSD_to_USDT_spot_price = Call(
    "0x390f3595bCa2Df7d23783dFd126427CCeb997BF4",
    ["get_dy(int128,int128,uint256)(uint256)", 1, 0, int(1e18)],
    [("crvUSD_to_USDT_spot_price", safe_normalize_6_with_bool_success)],
)


USDe_to_USDC_spot_price = Call(
    "0x02950460E2b9529D0E00284A5fA2d7bDF3fA4d72",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("USDe_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


sUSD_to_USDC_spot_price = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 3, 1, int(1e18)],
    [("sUSD_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


# balancer GHO -> USDC
# https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9

# https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9
# USDT, GHO, USDC


calls = [
    crvUSD_to_USDC_spot_price,
    USDe_to_USDC_spot_price,
    DAI_to_USDC_spot_price,
    USDT_to_USDC_spot_price,
]

get_state_by_one_block(calls, max(blocks), ETH_CHAIN)

NameError: name 'blocks' is not defined

In [ ]:
# s  # sUSDe -> usdc https://balancer.fi/pools/ethereum/v2/0xb819feef8f0fcdc268afe14162983a69f6bf179e000000000000000000000689

# gho, USDC, USDT tri pool  https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9

# https://curve.fi/dex/ethereum/pools/factory-crvusd-0/deposit/
# crvUSD -> USDC


# dai, USDC, USDT tri pool
# https://curve.fi/dex/ethereum/pools/3pool/deposit/

# USDe -> USDC https://curve.fi/dex/ethereum/pools/factory-stable-ng-12/deposit/

# USDs does not have good sorces of liquidty


# https://app.uniswap.org/explore/pools/ethereum/0x5777d92f208679DB4b9778590Fa3CAB3aC9e2168

In [ ]:
# from web3 import Web3

# # Assume you have an initialized Web3 instance (e.g. using an Infura or Alchemy endpoint)

# queries_address = Web3.toChecksumAddress("0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5")

# pool_id = bytes.fromhex("0xb819feef8f0fcdc268afe14162983a69f6bf179e000000000000000000000689"[2:])
# amount_in = int(1e18)
# swap_kind = 0
# user_data = b""

# from_internal_balance = False
# to_internal_balance = False

# call_obj = Call(
#     "0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5",  # Balancer Queries contract address
#     [
#         # Function signature for querySwap with two tuple parameters:
#         "querySwap((bytes32,uint8,address,address,uint256,bytes),(address,bool,address,bool))(uint256)",
#         pool_id,
#         swap_kind,
#         sUSDe,
#         USDC,
#         amount_in,
#         user_data,
#         '0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5',
#         from_internal_balance,
#         '0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5',
#         to_internal_balance
#     ]
#     [("usdc_out", safe_normalize_6_with_bool_success)]
# )

# get_state_by_one_block([call_obj],22027082, ETH_CHAIN )